# Shared LSC Context Audit

This notebook audits the shared mention-level context table created by `01_build_lsc_contexts.ipynb`. It does not rebuild contexts. Its role is to verify that the handoff table is structurally sound, surface coverage and domain-concentration risks, and prepare a compact manual-inspection handoff.

## Setup

The hard checks below catch problems that would make downstream LSC notebooks unsafe to run. Warning flags identify issues that need interpretation rather than immediate failure.

In [13]:
from pathlib import Path

import pandas as pd
import yaml

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs/commoncrawl_collection.yaml"
CONTEXT_DIR = PROJECT_ROOT / "data/interim/lsc/contexts"

CONTEXT_PATH = CONTEXT_DIR / "lsc_mention_contexts.parquet"
COUNTS_BY_UNIT_PATH = CONTEXT_DIR / "lsc_context_counts_by_year_unit.csv"
COUNTS_BY_RAW_PATH = CONTEXT_DIR / "lsc_context_counts_by_year_unit_raw_form.csv"
TOP_DOMAINS_PATH = CONTEXT_DIR / "lsc_context_top_domains_by_year_unit.csv"
MANUAL_SAMPLE_PATH = CONTEXT_DIR / "lsc_context_manual_samples.csv"
EXTRACTION_SUMMARY_PATH = CONTEXT_DIR / "lsc_context_extraction_summary.csv"
AUDIT_CHECKS_PATH = CONTEXT_DIR / "lsc_context_audit_checks.csv"
AUDIT_FLAGS_PATH = CONTEXT_DIR / "lsc_context_audit_flags.csv"

EXPECTED_UNITS = ["ADHD", "Autism", "frustration", "loneliness", "sadness"]
MAX_MENTIONS_PER_DOC_UNIT = 3
LOW_DOCUMENT_WARN_THRESHOLD = 50
TOP_DOMAIN_SHARE_WARN_THRESHOLD = 0.20
MIN_MANUAL_SAMPLES_PER_UNIT_YEAR = 1

with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

collection_window = config["collection"]["window"]
EXPECTED_YEARS = list(range(collection_window["primary_start_year"], collection_window["end_year"] + 1))

PROJECT_ROOT

PosixPath('/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak')

## Load Builder Outputs

Most diagnostics are read from the builder outputs. The parquet table is loaded with only the columns needed for structural checks, so the audit stays lighter than the context-building notebook.

In [14]:
required_paths = {
    "context_table": CONTEXT_PATH,
    "counts_by_unit": COUNTS_BY_UNIT_PATH,
    "counts_by_raw": COUNTS_BY_RAW_PATH,
    "top_domains": TOP_DOMAINS_PATH,
    "manual_samples": MANUAL_SAMPLE_PATH,
    "extraction_summary": EXTRACTION_SUMMARY_PATH,
}
missing_paths = [name for name, path in required_paths.items() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing LSC context output(s): {missing_paths}")

context_columns = [
    "doc_id",
    "source_year",
    "analysis_unit",
    "raw_form",
    "collapsed_raw_forms",
    "collapsed_match_count",
    "acronym_expansion_collapsed",
    "registered_domain",
    "mention_start_char",
    "mention_end_char",
    "target_sentence",
    "token_window_5",
    "cap_applied",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns)
counts_by_unit = pd.read_csv(COUNTS_BY_UNIT_PATH)
counts_by_raw = pd.read_csv(COUNTS_BY_RAW_PATH)
top_domains = pd.read_csv(TOP_DOMAINS_PATH)
manual_samples = pd.read_csv(MANUAL_SAMPLE_PATH)
extraction_summary = pd.read_csv(EXTRACTION_SUMMARY_PATH)

print(f"Context rows: {len(contexts):,}")
print(f"Documents: {contexts['doc_id'].nunique():,}")
print(f"Analysis units: {', '.join(sorted(contexts['analysis_unit'].dropna().unique()))}")

Context rows: 232,009
Documents: 167,506
Analysis units: ADHD, Autism, frustration, loneliness, sadness


## Normalise Diagnostic Tables

The builder now writes top-domain ranks and shares, but this cell also supports earlier top-domain files that contain only raw counts. That makes the audit robust when rerun after partial or interrupted preprocessing.

In [15]:
top_domains = top_domains.copy()
sort_columns = ["source_year", "analysis_unit", "mentions", "registered_domain"]
top_domains = top_domains.sort_values(sort_columns, ascending=[True, True, False, True]).reset_index(drop=True)

if "domain_rank" not in top_domains.columns:
    top_domains["domain_rank"] = top_domains.groupby(["source_year", "analysis_unit"]).cumcount() + 1

if "total_mentions_for_year_unit" not in top_domains.columns:
    denominators = counts_by_unit[["source_year", "analysis_unit", "mentions"]].rename(columns={"mentions": "total_mentions_for_year_unit"})
    top_domains = top_domains.merge(denominators, on=["source_year", "analysis_unit"], how="left")

if "mention_share" not in top_domains.columns:
    top_domains["mention_share"] = top_domains["mentions"] / top_domains["total_mentions_for_year_unit"]

top_domains.head()

,source_year,analysis_unit,registered_domain,mentions,documents,total_mentions_for_year_unit,mention_share,domain_rank
0,2014,ADHD,help4adhd.org,120,40,1774,0.067644,1
1,2014,ADHD,naturalnews.com,55,27,1774,0.031003,2
2,2014,ADHD,additudemag.com,49,17,1774,0.027621,3
3,2014,ADHD,bizofbaseball.com,21,14,1774,0.011838,4
4,2014,ADHD,mayoclinic.org,18,10,1774,0.010147,5


## Hard Structural Checks

In [16]:
checks: list[dict[str, object]] = []

def add_check(name: str, value: object, passes: bool, severity: str = "fail") -> None:
    checks.append({"check": name, "value": value, "passes": bool(passes), "severity": severity})

observed_units = sorted(contexts["analysis_unit"].dropna().unique())
observed_years = sorted(contexts["source_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
unexpected_units = sorted(set(observed_units) - set(EXPECTED_UNITS))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
unexpected_years = sorted(set(observed_years) - set(EXPECTED_YEARS))

max_mentions_per_doc_unit = int(contexts.groupby(["doc_id", "analysis_unit"]).size().max())
duplicate_mentions = int(contexts.duplicated(["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]).sum())
missing_target_sentence = int(contexts["target_sentence"].fillna("").str.strip().eq("").sum())
missing_token_window = int(contexts["token_window_5"].fillna("").str.strip().eq("").sum())
invalid_offsets = int((contexts["mention_end_char"] <= contexts["mention_start_char"]).sum())
missing_collapsed_raw_forms = int(contexts["collapsed_raw_forms"].fillna("").str.strip().eq("").sum())
missing_collapsed_match_count = int(contexts["collapsed_match_count"].isna().sum())
summary_metrics = dict(zip(extraction_summary["metric"], extraction_summary["value"]))
collapse_arithmetic_expected = int(summary_metrics.get("matches_after_overlap_resolution", 0)) - int(summary_metrics.get("acronym_expansion_mentions_removed", 0))
collapse_arithmetic_observed = int(summary_metrics.get("matches_after_acronym_expansion_collapse", -1))
unit_count_total = int(counts_by_unit["mentions"].sum())
raw_count_total = int(counts_by_raw["mentions"].sum())

add_check("context_rows_positive", len(contexts), len(contexts) > 0)
add_check("expected_analysis_units_present", ", ".join(observed_units), not missing_units)
add_check("no_unexpected_analysis_units", ", ".join(unexpected_units), not unexpected_units, severity="warn")
add_check("expected_years_present", ", ".join(map(str, observed_years)), not missing_years)
add_check("no_unexpected_years", ", ".join(map(str, unexpected_years)), not unexpected_years, severity="warn")
add_check("mention_cap_respected", max_mentions_per_doc_unit, max_mentions_per_doc_unit <= MAX_MENTIONS_PER_DOC_UNIT)
add_check("duplicate_mention_rows", duplicate_mentions, duplicate_mentions == 0)
add_check("target_sentence_non_empty", missing_target_sentence, missing_target_sentence == 0)
add_check("token_window_non_empty", missing_token_window, missing_token_window == 0)
add_check("valid_mention_offsets", invalid_offsets, invalid_offsets == 0)
add_check("collapsed_raw_forms_recorded", missing_collapsed_raw_forms, missing_collapsed_raw_forms == 0)
add_check("collapsed_match_count_recorded", missing_collapsed_match_count, missing_collapsed_match_count == 0)
add_check("collapse_summary_arithmetic", collapse_arithmetic_observed, collapse_arithmetic_observed == collapse_arithmetic_expected)
add_check("unit_counts_match_context_rows", unit_count_total, unit_count_total == len(contexts))
add_check("raw_counts_match_context_rows", raw_count_total, raw_count_total == len(contexts))

audit_checks = pd.DataFrame(checks)
audit_checks.to_csv(AUDIT_CHECKS_PATH, index=False)
audit_checks

,check,value,passes,severity
0,context_rows_positive,232009,True,fail
1,expected_analysis_units_present,"ADHD, Autism, frustration, loneliness, sadness",True,fail
2,no_unexpected_analysis_units,,True,warn
3,expected_years_present,"2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026",True,fail
4,no_unexpected_years,,True,warn
5,mention_cap_respected,3,True,fail
6,duplicate_mention_rows,0,True,fail
7,target_sentence_non_empty,0,True,fail
8,token_window_non_empty,0,True,fail
9,valid_mention_offsets,0,True,fail


## Coverage Diagnostics

Low sample sizes are warning flags, not automatic exclusions. Downstream notebooks should decide whether to pool, smooth, or report uncertainty for low-volume unit-years.

In [17]:
coverage_mentions = counts_by_unit.pivot(index="source_year", columns="analysis_unit", values="mentions").reindex(columns=EXPECTED_UNITS).fillna(0).astype(int)
coverage_documents = counts_by_unit.pivot(index="source_year", columns="analysis_unit", values="documents").reindex(columns=EXPECTED_UNITS).fillna(0).astype(int)

low_coverage = counts_by_unit.loc[counts_by_unit["documents"] < LOW_DOCUMENT_WARN_THRESHOLD, ["source_year", "analysis_unit", "documents", "mentions"]].copy()
coverage_mentions

analysis_unit,ADHD,Autism,frustration,loneliness,sadness
source_year,,,,,
2014,1774,4543,6449,1498,3014
2015,1695,4804,7403,1631,3338
2016,1790,5231,9845,2073,4484
2017,1547,4170,6597,1840,3400
2018,1886,5171,7665,2164,4514
2019,1835,4236,4883,1648,2763
2020,1498,3464,5254,1935,3043
2021,1729,4469,7042,2805,4016
2022,1711,4002,6432,2501,3867


In [18]:
coverage_documents

analysis_unit,ADHD,Autism,frustration,loneliness,sadness
source_year,,,,,
2014,997,2506,5701,1233,2515
2015,984,2750,6515,1320,2846
2016,1111,3039,8776,1741,3867
2017,987,2438,5853,1501,2868
2018,1185,3037,6871,1840,3860
2019,1169,2490,4358,1367,2356
2020,961,2105,4633,1566,2560
2021,1150,2698,6279,2245,3421
2022,1119,2383,5711,2037,3267


## Domain-Concentration Diagnostics

A high top-domain share can indicate genuine discourse concentration or residual crawler/template artefacts. These cases should be inspected before interpreting a year-specific spike.

In [19]:
top_domain_rows = top_domains.loc[top_domains["domain_rank"] == 1, ["source_year", "analysis_unit", "registered_domain", "mentions", "documents", "mention_share"]].copy()
domain_concentration_flags = top_domain_rows.loc[top_domain_rows["mention_share"] >= TOP_DOMAIN_SHARE_WARN_THRESHOLD].sort_values("mention_share", ascending=False)
top_domain_rows.sort_values("mention_share", ascending=False).head(20)

,source_year,analysis_unit,registered_domain,mentions,documents,mention_share
50,2015,ADHD,help4adhd.org,166,72,0.097935
130,2016,loneliness,astrotheme.com,141,87,0.068017
0,2014,ADHD,help4adhd.org,120,40,0.067644
80,2015,loneliness,astrotheme.com,101,62,0.061925
30,2014,loneliness,astrotheme.com,85,61,0.056742
100,2016,ADHD,additudemag.com,87,33,0.048603
180,2017,loneliness,astrotheme.com,75,50,0.040761
530,2024,loneliness,talkwithstranger.com,73,25,0.031330
60,2015,Autism,autismspeaks.org,147,51,0.030600
10,2014,Autism,autismspeaks.org,109,37,0.023993


## Raw-Form Balance

Raw-form totals are diagnostic only. The main target analysis remains at conceptual group level for ADHD and Autism.

In [20]:
raw_form_totals = (
    counts_by_raw.groupby(["analysis_unit", "raw_form"], as_index=False)
    .agg(documents=("documents", "sum"), mentions=("mentions", "sum"))
    .sort_values(["analysis_unit", "mentions"], ascending=[True, False])
)
raw_form_totals

,analysis_unit,raw_form,documents,mentions
0,ADHD,adhd,11957,17717
1,ADHD,attention_deficit,3533,4155
3,Autism,autism,24216,37428
5,Autism,autistic,7361,9251
4,Autism,autism_spectrum,5461,6551
2,Autism,asd_disambiguated,558,632
6,frustration,frustration,76148,86047
7,loneliness,loneliness,21872,27033
8,sadness,sadness,36607,43195


## Manual Sample Handoff

These rows are for qualitative inspection. They are not used to produce automatic exclusions in this notebook.

In [21]:
sample_counts = manual_samples.groupby(["source_year", "analysis_unit"]).size().reset_index(name="sample_rows")
expected_sample_keys = counts_by_unit[["source_year", "analysis_unit"]].drop_duplicates()
sample_coverage = expected_sample_keys.merge(sample_counts, on=["source_year", "analysis_unit"], how="left").fillna({"sample_rows": 0})
missing_samples = sample_coverage.loc[sample_coverage["sample_rows"] < MIN_MANUAL_SAMPLES_PER_UNIT_YEAR].copy()

sample_columns = [
    "source_year",
    "analysis_unit",
    "raw_form",
    "collapsed_raw_forms",
    "collapsed_match_count",
    "acronym_expansion_collapsed",
    "registered_domain",
    "target_sentence",
    "url",
]
manual_samples[sample_columns].head(30)

,source_year,analysis_unit,raw_form,collapsed_raw_forms,collapsed_match_count,acronym_expansion_collapsed,registered_domain,target_sentence,url
0,2014,ADHD,adhd,adhd,1,False,lnrmc.com,"Clinical Trials May Fall Into Federal Regulation 'Gap' Soy Won't Prevent Prostate Cancer's Return: Study Star Athletes Often Endorse Junk Food, Study Says Start at the Healthie...","http://www.lnrmc.com/health-education/all-related/6,679594/6"
1,2014,ADHD,adhd,adhd,1,False,koaa.com,"The study compared what's called the dopamine reward pathway in the brains of 53 adults who had ADHD with 44 adults who did not, using images taken at Brookhaven National Labor...",http://www.koaa.com/news/dopamine-seen-as-root-of-adhd1/
2,2014,ADHD,adhd,adhd,1,False,lawyersandsettlements.com,"com; 12/12/11), found no increased risk of heart attack or stroke in patients who use ADHD medications including Adderall.",http://www.lawyersandsettlements.com/articles/adderall/adderall-heart-attack-stroke-adhd-2-17242.html
3,2015,ADHD,adhd,adhd,1,False,nutritionexpress.com,"At the end of the study, kids in the Pycnogenol group had fewer ADHD symptoms and fewer signs of adrenaline and the nerve stimulant dopamine in the urine compared to the start ...",http://www.nutritionexpress.com/article+index/newsletters/2008+newsletters/october+2008/showarticle.aspx?id=990
4,2015,ADHD,attention_deficit,attention_deficit,1,False,4-adhd.com,The behaviors that indicate attention deficit hyperactivity disorder can vary from person to person.,http://www.4-adhd.com/
5,2015,ADHD,attention_deficit,attention_deficit,1,False,suigin-iranai.jp,Mercury is the most toxic non-radioactive element in the world Mercury poisoning can cause behavioral pathologies ranging from Asperger’s Syndrome and attention deficit hyperac...,http://suigin-iranai.jp/en
6,2016,ADHD,adhd,adhd,1,False,flinger.us,"The Universe is One Persistent Mofo Jul 12, 2014 Over a year ago, I started talking to someone about ADHD.",http://mrs.flinger.us/summer06/blog_permalink/im_the_seventh_grader_with_hairy_legs/contact.html/P10/
7,2016,ADHD,adhd,adhd,1,False,mrdad.com,It’s pretty widely accepted these days that too many young children—especially boys—are being diagnosed with ADHD.,https://mrdad.com/blog/ethics-and-dangers-of-unneeded-adhd-meds/
8,2016,ADHD,adhd,adhd,1,False,theatlantic.com,"So I couldn't help but feel a little glee upon discovering that Jason Fletcher, a professor at Yale's School of Public Health who has studied the way teenage depression and ADH...",http://www.theatlantic.com/business/archive/2013/07/revenge-of-the-nerds-being-popular-in-high-school-doesnt-make-you-rich-after-all/278076/
9,2017,ADHD,adhd,adhd,1,False,apples4theteacher.com,"Check availability Email this page to a friend Email this page to a friend ADD, ADHD, Literacy, ESL, Special Ed, Bilingual Ed, Gifted, Health Ed, Early Childhood Education Home...",http://www.apples4theteacher.com/holidays/winter/kids-crafts/snowflake-inch-worm.html


## Audit Flags and Verdict

In [22]:
flags: list[dict[str, object]] = []

for _, row in audit_checks.loc[~audit_checks["passes"]].iterrows():
    flags.append({"severity": row["severity"], "category": "structural_check", "source_year": None, "analysis_unit": None, "detail": row["check"], "value": row["value"]})

for _, row in low_coverage.iterrows():
    flags.append({"severity": "warn", "category": "low_document_count", "source_year": int(row["source_year"]), "analysis_unit": row["analysis_unit"], "detail": "documents below warning threshold", "value": int(row["documents"])})

for _, row in domain_concentration_flags.iterrows():
    flags.append({"severity": "warn", "category": "top_domain_concentration", "source_year": int(row["source_year"]), "analysis_unit": row["analysis_unit"], "detail": row["registered_domain"], "value": float(row["mention_share"])})

for _, row in missing_samples.iterrows():
    flags.append({"severity": "warn", "category": "manual_sample_missing", "source_year": int(row["source_year"]), "analysis_unit": row["analysis_unit"], "detail": "manual sample rows below minimum", "value": int(row["sample_rows"])})

audit_flags = pd.DataFrame(flags, columns=["severity", "category", "source_year", "analysis_unit", "detail", "value"])
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

hard_failures = audit_flags.loc[audit_flags["severity"] == "fail"]
if not hard_failures.empty:
    raise AssertionError(f"Shared LSC context audit failed hard checks: {hard_failures['detail'].tolist()}")

print(f"Wrote {AUDIT_CHECKS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {AUDIT_FLAGS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Warnings: {(audit_flags['severity'] == 'warn').sum() if not audit_flags.empty else 0}")
print("Shared LSC context audit passed hard checks.")
audit_flags.head(30)

Wrote data/interim/lsc/contexts/lsc_context_audit_checks.csv
Wrote data/interim/lsc/contexts/lsc_context_audit_flags.csv
Warnings: 0
Shared LSC context audit passed hard checks.


,severity,category,source_year,analysis_unit,detail,value
